---

### **01 ) -** Código

---

In [5]:
from kaggle.api.kaggle_api_extended import KaggleApi
import pandas as pd
import numpy as np
import kaggle
import shutil
import json
import os

In [6]:
kaggle.api.authenticate()

In [ ]:
KAGGLE_BASE_URL = "https://www.kaggle.com/datasets/"

In [39]:
def get_columns_size_mean(list_colums : list[str]) -> float:
    list_mean = []
    for col in list_colums:
        list_mean.append(len(col))
    return np.mean(list_mean)

def is_encrypted(x : float, threshold : int = 5) -> bool:
    if x < threshold:
        return True
    else:
        return False

In [43]:
def create_df(search_term='credit', max_datasets=1, path='./DataBuffer'):
    api = KaggleApi()
    api.authenticate()
    
    datasets = api.dataset_list(search=search_term)
    data_list = []

    if not os.path.exists(path):
        os.makedirs(path)

    # Limita o loop ao menor valor entre o solicitado e o disponível
    limit = min(max_datasets, len(datasets))

    for i in range(limit):
        api_data = datasets[i]
        ref = api_data.ref
        
        try:
            api.dataset_download_files(ref, path=path, unzip=True)
            files = [f for f in os.listdir(path) if f.endswith('.csv')]
            
            for file in files:
                file_path = os.path.join(path, file)
                
                # Leitura completa para extrair metadados de qualidade
                # Para arquivos extremamente grandes, considere usar amostragens
                df_temp = pd.read_csv(file_path)
                
                # Identificação de colunas sensíveis (comum em estudos de viés)
                sensitive_terms = ['age', 'gender', 'sex', 'race', 'ethnicity', 'status']
                found_sensitive = [col for col in df_temp.columns if any(s in col.lower() for s in sensitive_terms)]
                
                # Faz uma verificação inicial da criptografía
                columns = df_temp.columns.values

                data_list.append({
                    'User_Ref': ref,
                    'Dataset_Title': api_data.title,
                    'File_Name': file,
                    'URL': KAGGLE_BASE_URL + ref,
                    'Usability_Rating': getattr(api_data, 'usabilityRating', None),
                    'License': getattr(api_data, 'licenseName', 'Unknown'),
                    'is_encrypted' : is_encrypted(x=get_columns_size_mean(columns), threshold=5),
                    'Columns' : columns,
                    'Columns_Count': df_temp.shape[1],
                    'Rows_Count': df_temp.shape[0],
                    'Missing_Values': df_temp.isnull().sum().sum(),
                    'Numeric_Cols': len(df_temp.select_dtypes(include=['number']).columns),
                    'Categorical_Cols': len(df_temp.select_dtypes(exclude=['number']).columns),
                    'Sensitive_Columns': found_sensitive,
                    'Memory_Usage_MB': round(df_temp.memory_usage(deep=True).sum() / (1024**2), 2)
                })
                
        except Exception as e:
            print(f"Erro ao processar {ref}: {e}")
        
        finally:
            # Limpeza rigorosa do buffer
            for f in os.listdir(path):
                file_to_rem = os.path.join(path, f)
                try:
                    if os.path.isfile(file_to_rem):
                        os.remove(file_to_rem)
                    elif os.path.isdir(file_to_rem):
                        shutil.rmtree(file_to_rem)
                except Exception as cleanup_error:
                    print(f"Erro na limpeza: {cleanup_error}")

    return pd.DataFrame(data_list)

# Execução
df_info = create_df(search_term='credit', max_datasets=200)

Dataset URL: https://www.kaggle.com/datasets/sakshigoyal7/credit-card-customers
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
Dataset URL: https://www.kaggle.com/datasets/rikdifos/credit-card-approval-prediction
Dataset URL: https://www.kaggle.com/datasets/parisrohan/credit-score-classification


/tmp/ipykernel_4853/774472528.py:27: DtypeWarning: Columns (0: Monthly_Balance) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(file_path)


Dataset URL: https://www.kaggle.com/datasets/uciml/german-credit
Dataset URL: https://www.kaggle.com/datasets/dhanushnarayananr/credit-card-fraud
Dataset URL: https://www.kaggle.com/datasets/ranadeep/credit-risk-dataset
Dataset URL: https://www.kaggle.com/datasets/laotse/credit-risk-dataset
Dataset URL: https://www.kaggle.com/datasets/ealtman2019/credit-card-transactions
Dataset URL: https://www.kaggle.com/datasets/joebeachcapital/credit-card-fraud
Dataset URL: https://www.kaggle.com/datasets/mishra5001/credit-card
Erro ao processar mishra5001/credit-card: 'utf-8' codec can't decode byte 0x85 in position 1127: invalid start byte
Dataset URL: https://www.kaggle.com/datasets/ayushchandramaurya/credit-card-spendings
Dataset URL: https://www.kaggle.com/datasets/nelgiriyewithana/credit-card-fraud-detection-dataset-2023
Dataset URL: https://www.kaggle.com/datasets/whenamancodes/credit-card-customers-prediction
Dataset URL: https://www.kaggle.com/datasets/upadorprofzs/credit-risk
Dataset URL:

In [44]:
df_info

,User_Ref,Dataset_Title,File_Name,URL,Usability_Rating,License,is_encrypted,Columns,Columns_Count,Rows_Count,Missing_Values,Numeric_Cols,Categorical_Cols,Sensitive_Columns,Memory_Usage_MB
0,sakshigoyal7/credit-card-customers,Credit Card customers,BankChurners.csv,https://www.kaggle.com/datasets/sakshigoyal7/c...,None,Unknown,False,"[CLIENTNUM, Attrition_Flag, Customer_Age, Gend...",23,10127,0,17,6,"[Customer_Age, Gender, Marital_Status]",4.63
1,mlg-ulb/creditcardfraud,Credit Card Fraud Detection,creditcard.csv,https://www.kaggle.com/datasets/mlg-ulb/credit...,None,Unknown,True,"[Time, V1, V2, V3, V4, V5, V6, V7, V8, V9, V10...",31,284807,0,31,0,[],67.36
2,rikdifos/credit-card-approval-prediction,Credit Card Approval Prediction,credit_record.csv,https://www.kaggle.com/datasets/rikdifos/credi...,None,Unknown,False,"[ID, MONTHS_BALANCE, STATUS]",3,1048575,0,2,1,[STATUS],66.00
3,rikdifos/credit-card-approval-prediction,Credit Card Approval Prediction,application_record.csv,https://www.kaggle.com/datasets/rikdifos/credi...,None,Unknown,False,"[ID, CODE_GENDER, FLAG_OWN_CAR, FLAG_OWN_REALT...",18,438557,134203,10,8,"[CODE_GENDER, NAME_FAMILY_STATUS]",225.41
4,parisrohan/credit-score-classification,Credit score classification,train.csv,https://www.kaggle.com/datasets/parisrohan/cre...,None,Unknown,False,"[ID, Customer_ID, Month, Name, Age, SSN, Occup...",28,100000,60071,8,20,"[Age, Credit_History_Age]",120.56
5,parisrohan/credit-score-classification,Credit score classification,test.csv,https://www.kaggle.com/datasets/parisrohan/cre...,None,Unknown,False,"[ID, Customer_ID, Month, Name, Age, SSN, Occup...",27,50000,30053,8,19,"[Age, Credit_History_Age]",57.81
6,uciml/german-credit,German Credit Risk,german_credit_data.csv,https://www.kaggle.com/datasets/uciml/german-c...,None,Unknown,False,"[Unnamed: 0, Age, Sex, Job, Housing, Saving ac...",10,1000,577,5,5,"[Age, Sex]",0.29
7,dhanushnarayananr/credit-card-fraud,Credit Card Fraud,card_transdata.csv,https://www.kaggle.com/datasets/dhanushnarayan...,None,Unknown,False,"[distance_from_home, distance_from_last_transa...",8,1000000,0,8,0,[],61.04
8,laotse/credit-risk-dataset,Credit Risk Dataset,credit_risk_dataset.csv,https://www.kaggle.com/datasets/laotse/credit-...,None,Unknown,False,"[person_age, person_income, person_home_owners...",12,32581,4011,8,4,"[person_age, loan_status]",8.63
9,ealtman2019/credit-card-transactions,Credit Card Transactions,sd254_cards.csv,https://www.kaggle.com/datasets/ealtman2019/cr...,None,Unknown,False,"[User, CARD INDEX, Card Brand, Card Type, Card...",13,6146,0,6,7,[],2.51
